In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('survey.csv') 

In [5]:
print(df.shape)
df.info()
df.describe()

(1259, 27)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Timestamp                  1259 non-null   object
 1   Age                        1259 non-null   int64 
 2   Gender                     1259 non-null   object
 3   Country                    1259 non-null   object
 4   state                      744 non-null    object
 5   self_employed              1241 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 1

,Age
count,1.259000e+03
mean,7.942815e+07
std,2.818299e+09
min,-1.726000e+03
25%,2.700000e+01
50%,3.100000e+01
75%,3.600000e+01
max,1.000000e+11


In [6]:
print(df['Gender'].value_counts())

Gender
Male                                              615
male                                              206
Female                                            121
M                                                 116
female                                             62
F                                                  38
m                                                  34
f                                                  15
Make                                                4
Male                                                3
Woman                                               3
Cis Male                                            2
Man                                                 2
Female (trans)                                      2
Female                                              2
Trans woman                                         1
msle                                                1
male leaning androgynous                            1
Neuter               

In [7]:
def clean_gender_final(val):
    val = str(val).lower().strip()
    
    male_list = [
        'male', 'm', 'make', 'cis male', 'man', 'msle', 'mail', 'malr', 
        'maile', 'mal', 'cis man', 'male (cis)'
    ]
    
    female_list = [
        'female', 'f', 'woman', 'female (cis)', 'cis-female/femme', 
        'femail', 'cis female', 'femake', 'female (trans)', 'trans woman', 
        'trans-female'
    ]
    
    if val in male_list:
        return 'Male'
    elif val in female_list:
        return 'Female'
    else:
        return 'Other'

df['Gender_Cleaned'] = df['Gender'].apply(clean_gender_final)

In [8]:
print(df['Gender_Cleaned'].value_counts())

Gender_Cleaned
Male      990
Female    251
Other      18
Name: count, dtype: int64


In [9]:
df = df[df['Country'] == 'United States']

to_drop = [
    'Timestamp', 
    'Country', 
    'comments', 
    'mental_health_interview', 
    'phys_health_interview', 
    'phys_health_consequence', 
    'mental_vs_physical',
    'Gender'
]

df = df.drop(columns=to_drop)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 751 entries, 0 to 1258
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Age                        751 non-null    int64 
 1   state                      740 non-null    object
 2   self_employed              740 non-null    object
 3   family_history             751 non-null    object
 4   treatment                  751 non-null    object
 5   work_interfere             607 non-null    object
 6   no_employees               751 non-null    object
 7   remote_work                751 non-null    object
 8   tech_company               751 non-null    object
 9   benefits                   751 non-null    object
 10  care_options               751 non-null    object
 11  wellness_program           751 non-null    object
 12  seek_help                  751 non-null    object
 13  anonymity                  751 non-null    object
 14  leave         

In [11]:
gen_map = {'Yes': 1.0, 'No': 0.0, "Don't know": 0.5}

leave_map = {
    'Very easy': 1.0,
    'Somewhat easy': 0.75,
    "Don't know": 0.5,
    'Somewhat difficult': 0.25,
    'Very difficult': 0.0
}

df['benefits_n'] = df['benefits'].map(gen_map)
df['care_n'] = df['care_options'].map(gen_map)
df['wellness_n'] = df['wellness_program'].map(gen_map)
df['help_n'] = df['seek_help'].map(gen_map)
df['anon_n'] = df['anonymity'].map(gen_map)
df['leave_n'] = df['leave'].map(leave_map)

df = df.drop(columns=['benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave'])

score_cols = ['benefits_n', 'care_n', 'wellness_n', 'help_n', 'anon_n', 'leave_n']
df['support_score'] = df[score_cols].sum(axis=1)

In [12]:
df['support_score'].describe()

count    751.000000
mean       3.040280
std        1.372634
min        0.000000
25%        2.000000
50%        2.750000
75%        4.000000
max        6.000000
Name: support_score, dtype: float64

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 751 entries, 0 to 1258
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        751 non-null    int64  
 1   state                      740 non-null    object 
 2   self_employed              740 non-null    object 
 3   family_history             751 non-null    object 
 4   treatment                  751 non-null    object 
 5   work_interfere             607 non-null    object 
 6   no_employees               751 non-null    object 
 7   remote_work                751 non-null    object 
 8   tech_company               751 non-null    object 
 9   mental_health_consequence  751 non-null    object 
 10  coworkers                  751 non-null    object 
 11  supervisor                 751 non-null    object 
 12  obs_consequence            751 non-null    object 
 13  Gender_Cleaned             751 non-null    object 
 14

In [14]:
df['care_n'] = df['care_n'].fillna(0.5)
df['support_score'] = df[score_cols].sum(axis=1)

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 751 entries, 0 to 1258
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        751 non-null    int64  
 1   state                      740 non-null    object 
 2   self_employed              740 non-null    object 
 3   family_history             751 non-null    object 
 4   treatment                  751 non-null    object 
 5   work_interfere             607 non-null    object 
 6   no_employees               751 non-null    object 
 7   remote_work                751 non-null    object 
 8   tech_company               751 non-null    object 
 9   mental_health_consequence  751 non-null    object 
 10  coworkers                  751 non-null    object 
 11  supervisor                 751 non-null    object 
 12  obs_consequence            751 non-null    object 
 13  Gender_Cleaned             751 non-null    object 
 14

In [15]:
abbrev_to_full = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming'
}

df['state_full'] = df['state'].map(abbrev_to_full)

df = df.drop(columns='state')


df['state_full'].value_counts()


state_full
California        138
Washington         70
New York           56
Tennessee          45
Texas              44
Ohio               30
Pennsylvania       29
Oregon             29
Illinois           28
Indiana            27
Michigan           22
Minnesota          21
Massachusetts      20
Florida            15
North Carolina     14
Virginia           14
Georgia            12
Wisconsin          12
Missouri           12
Utah               10
Colorado            9
Alabama             8
Arizona             7
Maryland            7
Oklahoma            6
New Jersey          6
Kentucky            5
South Carolina      5
Connecticut         4
Iowa                4
Nevada              3
Vermont             3
South Dakota        3
Kansas              3
New Hampshire       3
Wyoming             2
New Mexico          2
Nebraska            2
West Virginia       1
Idaho               1
Mississippi         1
Rhode Island        1
Louisiana           1
Maine               1
Name: count, dtype: i

In [16]:
df['self_employed'] = df['self_employed'].fillna('No')
df = df.dropna(subset=['state_full'])
df['work_interfere'] = df['work_interfere'].fillna('Don\'t know')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 736 entries, 0 to 1258
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        736 non-null    int64  
 1   self_employed              736 non-null    object 
 2   family_history             736 non-null    object 
 3   treatment                  736 non-null    object 
 4   work_interfere             736 non-null    object 
 5   no_employees               736 non-null    object 
 6   remote_work                736 non-null    object 
 7   tech_company               736 non-null    object 
 8   mental_health_consequence  736 non-null    object 
 9   coworkers                  736 non-null    object 
 10  supervisor                 736 non-null    object 
 11  obs_consequence            736 non-null    object 
 12  Gender_Cleaned             736 non-null    object 
 13  benefits_n                 736 non-null    float64
 14

In [18]:
df['remote_n'] = df['remote_work'].map({'Yes': 1, 'No': 0})
df['tech_n'] = df['tech_company'].map({'Yes': 1, 'No': 0})
df['self_n'] = df['self_employed'].map({'Yes': 1, 'No': 0})

size_map = {"1-5": 1, "6-25": 2, "26-100": 3, "100-500": 4, "500-1000": 5, "More than 1000": 6}
df['size_n'] = df['no_employees'].map(size_map)

df['consequence_n'] = df['mental_health_consequence'].map({'No': 0, 'Maybe': 1, 'Yes': 2})

In [19]:
df.to_csv('cleaned_data.csv', index=False)